# Test de multimodalidad: dibujar a ciegas

**Lección 6 · Clase 5.4** — cerramos la clase con un experimento que es mitad benchmark, mitad asombro. En *Sparks of AGI* (Bubeck et al., 2023), el equipo de Microsoft le pidió a GPT-4 —un modelo que **jamás había visto una imagen**— *"draw me a unicorn"* en TikZ, el lenguaje de diagramas de LaTeX. El resultado, torpe y reconocible a la vez, se volvió icónico: el modelo dibujaba **a ciegas**, componiendo formas desde una idea puramente textual de cómo se ve el mundo.

Tres años después repetimos el test con los modelos actuales, como forma de medir su **integración con lo visual**: TikZ es texto → lo compilamos con LaTeX → lo rasterizamos → y al final cerramos el círculo haciendo que un modelo *con visión* **mire** los dibujos y los califique. Texto que genera imagen que evalúa un modelo de imagen: la multimodalidad completa, ida y vuelta.

| | |
|---|---|
| **El pipeline** | pedir TikZ → compilar (LaTeX) → rasterizar (PyMuPDF) — con los fracasos de compilación como parte del dato. |
| **El torneo** | 3 modelos × 4 dibujos: unicornio (el clásico), bicicleta (el test duro de coherencia estructural), un pingüino esquiando (tema de la clase) y una red neuronal (diagrama técnico). |
| **La grilla** | Todos los dibujos lado a lado. |
| **El juez visual** | `gpt-5-mini` con visión puntúa cada imagen contra lo pedido. |


In [ ]:
# Esta lección usa el entorno uv del README. Si la corres en Colab, descomenta las DOS líneas:
# %pip install -q openai==2.53.0 pymupdf==1.28.2 matplotlib==3.11.1 python-dotenv==1.2.2
# !apt-get install -y -qq texlive-latex-extra > /dev/null
from dotenv import load_dotenv
import os
import shutil

load_dotenv()

try:
    from google.colab import userdata  # type: ignore
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY", "")
    except Exception:
        pass
except Exception:
    pass

HAY_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))

# Motor LaTeX: tectonic (brew install tectonic) o pdflatex (MacTeX / texlive)
MOTOR_LATEX = shutil.which("tectonic") or shutil.which("pdflatex")
HAY_LATEX = MOTOR_LATEX is not None

print("OPENAI_API_KEY presente:", HAY_OPENAI)
print("Motor LaTeX:", MOTOR_LATEX or "no encontrado")
if not HAY_OPENAI:
    print("⚠️ Sin OPENAI_API_KEY el torneo se salta.")
if not HAY_LATEX:
    print("⚠️ Sin LaTeX el notebook muestra el código TikZ pero no lo puede renderizar.")
    print("   macOS: brew install tectonic   ·   Colab/Linux: apt-get install texlive-latex-extra")

from pathlib import Path

DIR_TIKZ = Path("outputs/tikz")
DIR_TIKZ.mkdir(parents=True, exist_ok=True)

## El pipeline: texto → imagen, con testigos

Tres funciones cortas. La honestidad del benchmark está en `compilar`: si el modelo emite TikZ inválido, **eso es un resultado** (❌), no una excepción que esconder — la coherencia sintáctica bajo presión espacial es parte de lo que medimos.

In [ ]:
import re
import subprocess

from openai import OpenAI

cliente = OpenAI() if HAY_OPENAI else None

PLANTILLA = r"""\documentclass[border=8pt]{standalone}
\usepackage{tikz}
%(librerias)s
\begin{document}
%(cuerpo)s
\end{document}
"""


def pedir_tikz(modelo: str, pedido: str) -> str:
    respuesta = cliente.chat.completions.create(
        model=modelo,
        messages=[{
            "role": "user",
            "content": (
                f"Dibuja {pedido} en TikZ. Devuelve SOLO el bloque desde "
                "\\begin{tikzpicture} hasta \\end{tikzpicture} (más las líneas "
                "\\usetikzlibrary que necesites). Sin explicaciones, sin markdown."
            ),
        }],
    )
    return respuesta.choices[0].message.content or ""


def limpiar_codigo(crudo: str) -> str:
    """Quita fences de markdown y texto suelto alrededor del TikZ."""
    texto = re.sub(r"```[a-z]*\n?", "", crudo).strip()
    inicio = texto.find(r"\begin{tikzpicture}")
    fin = texto.rfind(r"\end{tikzpicture}")
    cuerpo = texto[inicio : fin + len(r"\end{tikzpicture}")] if inicio != -1 and fin != -1 else texto
    librerias = "\n".join(dict.fromkeys(re.findall(r"\\usetikzlibrary\{[^}]*\}", texto)))
    return PLANTILLA % {"librerias": librerias, "cuerpo": cuerpo}


def compilar_y_rasterizar(tex: str, nombre: str) -> Path | None:
    """Compila el .tex y lo rasteriza a PNG. None = no compiló (dato, no error)."""
    ruta_tex = DIR_TIKZ / f"{nombre}.tex"
    ruta_tex.write_text(tex)
    if "tectonic" in MOTOR_LATEX:
        comando = [MOTOR_LATEX, "--outdir", str(DIR_TIKZ), str(ruta_tex)]
    else:
        comando = [MOTOR_LATEX, "-interaction=nonstopmode", "-halt-on-error",
                   "-output-directory", str(DIR_TIKZ), str(ruta_tex)]
    try:
        proceso = subprocess.run(comando, capture_output=True, timeout=90)
    except subprocess.TimeoutExpired:
        return None  # un LaTeX colgado también es un ✗
    ruta_pdf = ruta_tex.with_suffix(".pdf")
    if proceso.returncode != 0 or not ruta_pdf.exists():
        return None

    import pymupdf

    pagina = pymupdf.open(ruta_pdf)[0]
    ruta_png = ruta_tex.with_suffix(".png")
    pagina.get_pixmap(dpi=160).save(ruta_png)
    return ruta_png


def evaluar_dibujo(modelo: str, clave: str, pedido: str) -> dict:
    crudo = pedir_tikz(modelo, pedido)
    tex = limpiar_codigo(crudo)
    png = compilar_y_rasterizar(tex, f"{modelo.replace('.', '_')}__{clave}") if HAY_LATEX else None
    return {"modelo": modelo, "clave": clave, "pedido": pedido,
            "tex": tex, "png": png, "compilo": png is not None}

## El torneo

Tres modelos con una gradiente deliberada de capacidad (el flagship actual, un medio y un chico de la generación anterior) y cuatro pedidos con dificultades distintas: el unicornio mide composición orgánica; la **bicicleta** es el test clásico de coherencia estructural (dos ruedas, marco que conecte, pedales donde van — es notablemente difícil); el pingüino esquiando mezcla figura y contexto; la red neuronal es un diagrama técnico, terreno natural de TikZ.

In [ ]:
MODELOS = ["gpt-5.5", "gpt-5.4-mini", "gpt-5-nano"]

DIBUJOS = {
    "unicornio": "un unicornio (el clásico de Sparks of AGI)",
    "bicicleta": "una bicicleta vista de lado, estructuralmente coherente",
    "pinguino": "un pingüino esquiando montaña abajo",
    "red_neuronal": "una red neuronal de 3 capas (4, 6 y 2 neuronas) con conexiones",
}

resultados = []
if HAY_OPENAI and HAY_LATEX:
    for modelo in MODELOS:
        for clave, pedido in DIBUJOS.items():
            resultado = evaluar_dibujo(modelo, clave, pedido)
            resultados.append(resultado)
            print(f"  {modelo:<14} {clave:<14} {'✓ compiló' if resultado['compilo'] else '✗ NO compiló'}")
elif HAY_OPENAI:
    # sin LaTeX: al menos vemos el código que generaría un modelo
    ejemplo = pedir_tikz(MODELOS[-1], DIBUJOS["unicornio"])
    print("Sin motor LaTeX no podemos renderizar. El TikZ crudo de", MODELOS[-1], "para el unicornio:\n")
    print(ejemplo[:800])
else:
    print("⛔ Falta OPENAI_API_KEY.")

## La grilla

Cada fila un modelo, cada columna un pedido. Los ❌ son tan informativos como los dibujos: son modelos que perdieron la coherencia sintáctica intentando la coherencia espacial.

In [ ]:
if resultados:
    import matplotlib.pyplot as plt
    import matplotlib.image as mpimg

    figura, ejes = plt.subplots(len(MODELOS), len(DIBUJOS), figsize=(4 * len(DIBUJOS), 3.6 * len(MODELOS)))
    for i, modelo in enumerate(MODELOS):
        for j, clave in enumerate(DIBUJOS):
            eje = ejes[i][j]
            resultado = next(r for r in resultados if r["modelo"] == modelo and r["clave"] == clave)
            if resultado["compilo"]:
                eje.imshow(mpimg.imread(resultado["png"]))
            else:
                eje.text(0.5, 0.5, "✗\nno compiló", ha="center", va="center", fontsize=16, color="#c0392b")
            eje.set_xticks([]), eje.set_yticks([])
            if i == 0:
                eje.set_title(clave, fontsize=11)
            if j == 0:
                eje.set_ylabel(modelo, fontsize=10)
    plt.suptitle("Dibujar a ciegas: TikZ generado por texto, sin ver jamás el resultado", y=1.0)
    plt.tight_layout()
    plt.savefig("outputs/grilla_multimodalidad.png", dpi=110, bbox_inches="tight")
    plt.show()

## El juez visual: cerrar el círculo

Hasta acá el modelo dibujó sin ver. Ahora un modelo **con visión** mira cada PNG y lo puntúa contra el pedido — el mismo proveedor, la otra dirección de la multimodalidad. (Y sí: un LLM juzgando dibujos de LLMs tiene sus sesgos; para un ranking serio harían falta rúbricas y humanos. Como cierre de clase, alcanza y sobra.)

In [ ]:
import base64
import json as json_lib

if resultados:
    def juzgar(resultado: dict) -> dict:
        if not resultado["compilo"]:
            return {"puntaje": 0, "comentario": "no compiló"}
        imagen_b64 = base64.b64encode(Path(resultado["png"]).read_bytes()).decode()
        respuesta = cliente.chat.completions.create(
            model="gpt-5-mini",
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": (
                        f"Esta imagen debería ser: {resultado['pedido']}. "
                        "Puntúa de 1 a 5 qué tan reconocible y bien logrado está. "
                        "Devuelve SOLO un JSON: {\"puntaje\": n, \"comentario\": \"...\"} (comentario ≤ 12 palabras)."
                    )},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{imagen_b64}"}},
                ],
            }],
            response_format={"type": "json_object"},
        )
        return json_lib.loads(respuesta.choices[0].message.content)

    print(f"{'modelo':<14} {'dibujo':<14} {'puntaje':>7}  comentario")
    print("-" * 70)
    puntajes = {}
    for resultado in resultados:
        veredicto = juzgar(resultado)
        puntajes.setdefault(resultado["modelo"], []).append(veredicto["puntaje"])
        print(f"{resultado['modelo']:<14} {resultado['clave']:<14} {veredicto['puntaje']:>7}  {veredicto.get('comentario', '')}")

    print("\n── ranking (promedio sobre 4 dibujos) ──")
    for modelo, notas in sorted(puntajes.items(), key=lambda kv: -sum(kv[1]) / len(kv[1])):
        print(f"  {modelo:<14} {sum(notas) / len(notas):.2f}")

## Lectura de los resultados

Tres cosas suelen aparecer (tus corridas variarán — los modelos no son deterministas):

1. **La gradiente existe y se ve.** El flagship compone figuras más coherentes y falla menos al compilar; el modelo chico produce TikZ válido pero dibujos más esquemáticos. La capacidad "visual" de un modelo de texto escala con la capacidad general — es conocimiento del mundo, no un módulo aparte.
2. **La bicicleta sigue siendo dura.** Ruedas que no tocan el marco, pedales flotantes: la coherencia *estructural* (las partes correctas, conectadas correctamente) es más difícil que la silueta. Igual que en 2023 — solo que ahora falla más elegante.
3. **El diagrama técnico les sale mejor que el unicornio.** La red neuronal es el terreno donde TikZ abunda en el entrenamiento. La distancia entre "dibujar lo que hay en internet" y "dibujar lo que imaginas" es exactamente lo que este test hace visible.

Y el punto de la clase: en una sola lección pasaron **texto → código → imagen → juicio visual**, cuatro modalidades integradas con dos llamadas de API y un compilador de los años 80. Las integraciones no siempre son protocolos nuevos; a veces son tuberías bien elegidas entre cosas que ya existían.

*Referencias: [Sparks of AGI (Bubeck et al., 2023)](https://arxiv.org/abs/2303.12712) — sección 2.2, el unicornio; [PGF/TikZ](https://tikz.dev).*